### Imports

In [1]:
! apt install python3.10-venv -y -q
!python3 -m venv venv
!source venv/bin/activate

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  python3-pip-whl python3-setuptools-whl
The following NEW packages will be installed:
  python3-pip-whl python3-setuptools-whl python3.10-venv
0 upgraded, 3 newly installed, 0 to remove and 46 not upgraded.
Need to get 2473 kB of archives.
After this operation, 2882 kB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-pip-whl all 22.0.2+dfsg-1ubuntu0.3
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-setuptools-whl all 59.6.0-1.2ubuntu0.22.04.1 [788 kB]
Err:1 http://security.ubuntu.com/ubuntu jammy-updates/universe amd64 python3-pip-whl all 22.0.2+dfsg-1ubuntu0.3
  404  Not Found [IP: 185.125.190.39 80]
Ign:3 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 python3.10-venv amd64 3.10.12-1~22.04.2
Err:3 http://security.ubuntu.com/ubuntu jammy-updates

In [18]:
import os
import json
import base64
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from skimage.color import rgb2gray
from skimage.measure import label, regionprops
from skimage.util import img_as_ubyte
from numpy import pad
import requests
from PIL import Image
from numpy import asarray
import cv2
from skimage.transform import resize
from skimage.util import view_as_blocks
from skimage import io, transform, util, img_as_ubyte
from skimage.transform import resize
from skimage.util import view_as_blocks
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
import pandas as pd
import zipfile
from scipy import spatial
import skimage.util

### Datasets

In [8]:
images_dir = '/kaggle/input/ihc-cd8-img-2/Auploader/'
output_dir = '/kaggle/working/'
filename = '4_Lenti-HPV-07_CD8.tif'

In [9]:
res = requests.post(
    url='https://deepliif.org/api/infer',
    files={
        'img': open(f'{images_dir}/{filename}', 'rb')
    },
    # optional param that can be 10x, 20x (default) or 40x
    params={
        'resolution': '20x'
    }
)

data = res.json()

def b64_to_pil(b):
    return Image.open(BytesIO(base64.b64decode(b.encode())))

for name, img in data['images'].items():
    output_filepath = f'{output_dir}/{os.path.splitext(filename)[0]}_{name}.png'
    with open(output_filepath, 'wb') as f:
        b64_to_pil(img).save(f, format='PNG')

print(json.dumps(data['scoring'], indent=2))

{
  "num_neg": 2147,
  "num_pos": 1181,
  "num_total": 3328,
  "percent_pos": 35.5,
  "prob_thresh": 80,
  "size_thresh": 50
}


In [10]:
!ls -F

4_Lenti-HPV-07_CD8_DAPI.png    4_Lenti-HPV-07_CD8_Seg.png
4_Lenti-HPV-07_CD8_Hema.png    4_Lenti-HPV-07_CD8_SegOverlaid.png
4_Lenti-HPV-07_CD8_Lap2.png    4_Lenti-HPV-07_CD8_SegRefined.png
4_Lenti-HPV-07_CD8_Marker.png  venv/


### Processing

In [ ]:
colors = {'immune': [[200, 0, 0], [255, 13, 13]], 'tumor': [[0, 0, 225], [13, 13, 255]], 'normal': [[120, 120, 120], [135, 135, 135]]}
image = io.imread('/kaggle/working/4_Lenti-HPV-07_CD8_SegOverlaid.png')
resized_image = cv2.resize(image, (100, 100), interpolation = cv2.INTER_AREA)
resized_image_gray = rgb2gray(resized_image)
labeled_tumor = np.zeros_like(resized_image_gray, dtype=bool)
labeled_immune = np.zeros_like(resized_image_gray, dtype=bool)
labeled_normal = np.zeros_like(resized_image_gray, dtype=bool)
for channel in range(3):
    labeled_tumor = np.logical_or(labeled_tumor, np.logical_and(resized_image[..., channel] >= colors['tumor'][0][channel], resized_image[..., channel] <= colors['tumor'][1][channel]))
    labeled_immune = np.logical_or(labeled_immune, np.logical_and(resized_image[..., channel] >= colors['immune'][0][channel], resized_image[..., channel] <= colors['immune'][1][channel]))
    labeled_normal = np.logical_or(labeled_normal, np.logical_and(resized_image[..., channel] >= colors['normal'][0][channel], resized_image[..., channel] <= colors['normal'][1][channel]))
tumor_density = img_as_ubyte(labeled_tumor)
immune_density = img_as_ubyte(labeled_immune)
normal_density = img_as_ubyte(labeled_normal)
tumor_density = tumor_density // 255
immune_density = immune_density // 255
normal_density = normal_density // 255
io.imsave(f'{output_dir}/resized_immune.png', immune_density)
io.imsave(f'{output_dir}/resized_tumor.png', tumor_density)
io.imsave(f'{output_dir}/resized_normal.png', normal_density)

### For Tumor cells

In [15]:
tumor_img = io.imread('/kaggle/working/resized_tumor.png')
tumor_img = view_as_blocks(tumor_img, block_shape=(100, 100))
tumor_img = tumor_img.reshape(-1, 100, 100)
tumor_img = tumor_img.ravel()
tumor_img = tumor_img.astype(np.uint8)
np.savetxt('/kaggle/working/Tum_dense.csv', tumor_img, delimiter=',', fmt='%d', header='Tum_dense', comments='')
tumor_data = pd.read_csv("/kaggle/working/Tum_dense.csv")
proliferating_cells_condition = (tumor_data["Tum_dense"] > 0)
tumor_img = np.zeros_like(tumor_data["Tum_dense"])
tumor_img[proliferating_cells_condition] = np.random.randint(0, 10, np.sum(proliferating_cells_condition))
tumor_img = np.where(tumor_img > 0, 1, 0)
np.savetxt('/kaggle/working/Tum_prolif.csv', tumor_img, delimiter=',', fmt='%d', header='Tum_prolif', comments='')
tumor_img = io.imread('/kaggle/working/resized_tumor.png')
tumor_img = view_as_blocks(tumor_img, block_shape=(100, 100))
df = pd.DataFrame(tumor_img.ravel())
df = df[df[0] >= 0]
df = df.reset_index(drop=True)
df = df.reset_index()
df = df.rename(columns={'index': 'ID'})
df = df.drop(columns=[0])
df.to_csv('/kaggle/working/tumor_id.csv', index=False, header='id')

### For Immune (CD8) cells

In [16]:
immune_img = io.imread('/kaggle/working/resized_immune.png')
immune_img = view_as_blocks(immune_img, block_shape=(100, 100))
df = pd.DataFrame(immune_img.ravel())
df = df[df[0] >= 0]
df = df.reset_index(drop=True)
df = df.reset_index()
df = df.rename(columns={'index': 'ID'})
df = df.drop(columns=[0])
df.to_csv('/kaggle/working/immune_id.csv', index=False, header='id')
immune_img = immune_img.ravel()
immune_img = immune_img.astype(np.uint8)
np.savetxt('/kaggle/working/CD8_dense.csv', immune_img, delimiter=',', fmt='%d', header='CD8_dense', comments='')
immune_data = pd.read_csv("/kaggle/working/CD8_dense.csv")
proliferating_cells_condition = (immune_data["CD8_dense"] > 0)
immune_img = np.zeros_like(immune_data["CD8_dense"])
immune_img[proliferating_cells_condition] = np.random.randint(0, 8, np.sum(proliferating_cells_condition))
immune_img = np.where(immune_img > 0, 1, 0)
np.savetxt('/kaggle/working/CD8_prolif.csv', immune_img, delimiter=',', fmt='%d', header='CD8_prolif', comments='')
immune_data_2 = pd.read_csv("/kaggle/working/CD8_prolif.csv")
non_prolif_condition = (immune_data_2["CD8_prolif"] == 0) & proliferating_cells_condition
immune_img = np.where(non_prolif_condition, 1, 0)
np.savetxt('/kaggle/working/CD8_non_prolif.csv', immune_img, delimiter=',', fmt='%d', header='CD8_non_prolif', comments='')

### Compute RDF

In [19]:
immune_density = io.imread('/kaggle/working/resized_immune.png')
if np.max(immune_density) == 0:
    raise ValueError("Immune density image is all 0s")
tile_size = 20 
radii = np.arange(0, 10)
# Tile image and filter valid tiles
tiles = skimage.util.view_as_windows(immune_density, (tile_size, tile_size))
valid_tiles = [tile for tile in tiles if 10 <= np.sum(tile) <= 10000]
if not valid_tiles:
    raise ValueError("No valid immune tiles")
rdf = np.zeros((len(valid_tiles), len(radii) - 1))
for i, tile in enumerate(valid_tiles):
    rep_tile = np.tile(tile, (3, 3))
    coords = np.column_stack(np.where(rep_tile))
    # KNN 
    tree = spatial.cKDTree(coords)
    distances, _ = tree.query(coords, k=10)
    distances = distances[distances > 0]
    # Histogram with valid range    
    hist, bin_edges = np.histogram(distances, bins=radii, range=(0, 10))
    areas = np.pi * (bin_edges[1:]**2 - bin_edges[:-1]**2)
    expected = areas * np.sum(tile) * (np.sum(tile) - 1)
    expected[expected == 0] = np.nan
    rdf[i, :] = hist / expected
# Average
rdf_avg = np.nanmean(rdf, axis=0)

In [23]:
print(rdf_avg)

[0.00000000e+00 1.15355002e-03 1.86584920e-04 1.13060495e-05
 4.17490365e-07 1.74451156e-08 3.29776977e-10 7.51291859e-11
 0.00000000e+00]


### Compute SAM and VarSAM

In [20]:
def computeSAM(rdf_sim, rdf_obs, obs_range_tol=0.2, frac_within_tol=0.7):
    # rdf_sim and rdf_obs are RDF vectors for the simulated and observed images
    num_dists = len(rdf_obs)
    # Calculate acceptable range for each distance
    max_obs = np.maximum(rdf_obs + obs_range_tol * np.ptp(rdf_obs), 0)  
    min_obs = np.maximum(np.minimum(rdf_obs - obs_range_tol * np.ptp(rdf_obs), 0), 0)
    # Check if simulated RDF is within tolerance range
    within_tol = (rdf_sim > min_obs) & (rdf_sim < max_obs) 
    # Calculate fraction of dists where sim RDF is within tol 
    frac_within = np.sum(within_tol) / num_dists
    # SAM is fraction of dists above threshold 
    if frac_within >= frac_within_tol:
        sam = 1
    else:
        sam = 0 
    return sam, frac_within

def computeVarSAM(rdf_sim, rdf_obs):
    # Extract RDFs within first 15 distances
    rdf_sim_near = rdf_sim[:15]
    rdf_obs_near = rdf_obs[:15]
    # Calculate ranges
    range_sim = np.ptp(rdf_sim_near)  
    range_obs = np.ptp(rdf_obs_near)
    # VarSAM is the ratio of the ranges 
    # Capped between 0 and 1
    if range_sim > range_obs:
        var_sam = range_obs / range_sim
    else:
        var_sam = range_sim / range_obs
    var_sam = np.minimum(1, var_sam)
    var_sam = np.maximum(0, var_sam)
    return var_sam

In [21]:
sam, frac_within = computeSAM(rdf_avg, rdf_avg)

var_sam = computeVarSAM(rdf_avg, rdf_avg)

In [22]:
print(sam, frac_within, var_sam)

1 0.7777777777777778 1.0
